## Baseline comparison

This notebook runs a first benchmark on the official FI-2010 CF_1 train/test split using horizon 3.

Majority baseline: accuracy of 0.306, macro F1 of 0.156.

Logistic regression: accuracy of 0.439, macro F1 of 0.431.

HistGradientBoosting: accuracy of 0.522, macro F1 of 0.508.

In [10]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from lob_forecasting.data import load_fi2010_split
from lob_forecasting.evaluation import evaluate_predictions

In [11]:
X_train, y_train_all, X_test, y_test_all = load_fi2010_split(
    project_root=PROJECT_ROOT,
    market="NoAuction",
    normalization="Zscore",
    cf=1,
    verbose=True,
)

X_train: (39512, 144)
y_train_all: (39512, 5)
X_test: (38397, 144)
y_test_all: (38397, 5)


In [12]:
target_col = 3

y_train = y_train_all[:, target_col].astype(int)
y_test = y_test_all[:, target_col].astype(int)

In [13]:
results = []

In [14]:
majority_class = np.bincount(y_train).argmax()
y_pred_majority = np.full_like(y_test, fill_value=majority_class)

results.append(evaluate_predictions(y_test, y_pred_majority, "Majority Baseline"))

Majority Baseline
Accuracy: 0.3060
Macro F1: 0.1562



              precision    recall  f1-score   support

           1       0.00      0.00      0.00     13764
           2       0.31      1.00      0.47     11749
           3       0.00      0.00      0.00     12884

    accuracy                           0.31     38397
   macro avg       0.10      0.33      0.16     38397
weighted avg       0.09      0.31      0.14     38397



/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d

In [15]:
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
)

logreg.fit(X_train, y_train)
y_pred_logreg = logreg.predict(X_test)

results.append(evaluate_predictions(y_test, y_pred_logreg, "LogisticRegression"))

LogisticRegression
Accuracy: 0.4393
Macro F1: 0.4306

              precision    recall  f1-score   support

           1       0.49      0.40      0.44     13764
           2       0.41      0.32      0.36     11749
           3       0.42      0.59      0.49     12884

    accuracy                           0.44     38397
   macro avg       0.44      0.44      0.43     38397
weighted avg       0.44      0.44      0.43     38397



In [16]:
hgb = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42,
)

hgb.fit(X_train, y_train)
y_pred_hgb = hgb.predict(X_test)

results.append(evaluate_predictions(y_test, y_pred_hgb, "HistGradientBoosting"))

HistGradientBoosting
Accuracy: 0.5220
Macro F1: 0.5081

              precision    recall  f1-score   support

           1       0.53      0.60      0.56     13764
           2       0.61      0.32      0.42     11749
           3       0.49      0.63      0.55     12884

    accuracy                           0.52     38397
   macro avg       0.54      0.51      0.51     38397
weighted avg       0.54      0.52      0.51     38397



In [17]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("macro_f1", ascending=False)
results_df

,model,accuracy,macro_f1
2,HistGradientBoosting,0.522046,0.508099
1,LogisticRegression,0.439331,0.430611
0,Majority Baseline,0.305987,0.156197


In [18]:
RESULTS_DIR = PROJECT_ROOT / "experiments"
RESULTS_DIR.mkdir(exist_ok=True)

output_path = RESULTS_DIR / "baseline_results_cf1_horizon3.csv"
results_df.to_csv(output_path, index=False)

print(output_path)

/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/experiments/baseline_results_cf1_horizon3.csv
